# Experiment 5: Multi-Agent vs Single-Agent

## The problem
A single agent doing a multi-part task accumulates ALL exploration for every
part in one window. It bloats with detail the final answer doesn't need.

## The fix
Sub-agents (Section 8.3): a parent delegates each sub-task to a sub-agent with
its OWN context window. Each returns only a short summary. The parent's context
stays small -- context compression by ARCHITECTURE, not summarization.

## Method
A 3-part comparison task. Naive: one agent does it all. Engineered: a parent
calls 3 sub-agents and composes their summaries. Compare the parent/single
context peak (real per-call input tokens).

## Setup

In [1]:
import sys, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from core.settings import settings

TASK = ("Compare how three areas handle long-context problems: "
        "(1) attention/architecture, (2) position effects, (3) retrieval. "
        "Cover the relevant paper(s) for each area.")
SINGLE_PAPERS = ["1706.03762", "2307.03172", "2404.06654", "2005.11401"]
print("task ready |", len(SINGLE_PAPERS), "papers for the single agent")

task ready | 4 papers for the single agent


## One agent-runner

`peak_input_tokens` = the largest single call's inputTokens, straight from the
model's usage. Context grows monotonically, so that IS the peak window size --
a real number, not an estimate.

In [2]:
from strands import Agent
from strands.models import BedrockModel
from tools.load_document import load_document
from agents.subagents import (run_all_specialists, reset_usage, get_usage,
                              peak_input_tokens)

def run_agent(system_prompt, user_prompt, tools):
    """Run one agent; return (answer_text, real_peak_tokens, latency_s)."""
    model = BedrockModel(model_id=settings.bedrock_model_id,
                         region_name=settings.aws_region,
                         temperature=settings.temperature,
                         max_tokens=settings.output_max_tokens)
    agent = Agent(model=model, tools=tools, system_prompt=system_prompt,
                  callback_handler=None)
    start = time.time()
    result = agent(user_prompt)
    latency = time.time() - start
    text = "".join(b.get("text", "") for b in result.message.get("content", [])
                   if "text" in b)
    return text, peak_input_tokens(result), latency

## Naive -- one agent loads everything into one window

In [3]:
def run_single_agent():
    system = ("You are a research analyst. Use load_document to read ALL of these "
              f"papers: {SINGLE_PAPERS}. Then write a structured comparison of how "
              "three areas handle long-context problems: (1) attention/architecture, "
              "(2) position effects, (3) retrieval. Cite paper ids.")
    text, peak, latency = run_agent(system, TASK, [load_document])
    return {"answer": text, "peak_tokens": peak, "latency": latency}

In [4]:
single = run_single_agent()
print(f"Peak context tokens: {single['peak_tokens']:,}")
print(f"Latency: {single['latency']:.1f}s\n")
print(single["answer"][:400])

Peak context tokens: 80,059
Latency: 38.1s

Based on the analysis of these papers, here's a structured comparison of how three areas handle long-context problems:

1. Attention/Architecture
- Transformer (1706.03762): 
  - Introduced self-attention mechanism that allows direct connections between all positions in a sequence
  - Multi-head attention enables attending to different representation subspaces
  - Constant number of operations bet


## Engineered -- 3 isolated specialists + a composer

We run the three specialists explicitly (each in its own window) and let a
composer write the final comparison from ONLY their summaries. Deterministic
delegation -- Haiku as an autonomous orchestrator sometimes answers from its own
knowledge and skips the sub-agents, which would silently invalidate the run.

In [5]:
import time

def run_multi_agent():
    reset_usage()
    start = time.time()
    specialists = run_all_specialists(TASK)        # 3 isolated windows, guaranteed
    blocks = "\n\n".join(f"[{s['name']}] {s['area']}:\n{s['summary']}"
                          for s in specialists)
    prompt = (f"{TASK}\n\nWrite one structured comparison of the three areas using "
              f"ONLY these specialist summaries:\n\n{blocks}")
    system = ("You are a research coordinator. Compose the final comparison from the "
              "specialist summaries you are given; use only what they provide.")
    text, peak, _ = run_agent(system, prompt, tools=[])   # composer sees only summaries
    usage = get_usage()
    return {"answer": text, "parent_peak_tokens": peak, "subagent_usage": usage,
            "subagent_tokens": sum(u["peak_tokens"] for u in usage),
            "latency": time.time() - start}

In [6]:
multi = run_multi_agent()
print(f"Parent peak tokens: {multi['parent_peak_tokens']:,}")
for u in multi["subagent_usage"]:
    print(f"  sub-agent [{u['name']}] peaked at {u['peak_tokens']:,} tokens (ISOLATED)")
print(f"Latency: {multi['latency']:.1f}s\n")
print(multi["answer"][:400])

Parent peak tokens: 1,056
  sub-agent [attention] peaked at 11,942 tokens (ISOLATED)
  sub-agent [position] peaked at 47,300 tokens (ISOLATED)
  sub-agent [retrieval] peaked at 21,942 tokens (ISOLATED)
Latency: 69.7s

Comparative Analysis of Long-Context Problem Handling Approaches

1. Attention/Architecture Approach
- Fundamental Strategy: Parallel, constant-time self-attention processing
- Key Mechanisms:
  * Multi-head attention for nuanced dependency capture
  * Constant path length between input/output positions
  * Sinusoidal positional encodings to preserve sequence order
- Strengths: Computational effic


## Comparison

In [7]:
print(f"Single-agent peak context:   {single['peak_tokens']:,} tokens")
print(f"Multi-agent PARENT context:  {multi['parent_peak_tokens']:,} tokens")
print(f"(sub-agent work stayed isolated, never in parent: "
      f"{multi['subagent_tokens']:,} tokens total)")
shrink = 100 * (1 - multi['parent_peak_tokens'] / single['peak_tokens'])
print(f"Parent window is {shrink:.0f}% smaller than the single agent's.\n")
print(f"Single latency:  {single['latency']:.1f}s")
print(f"Multi latency:   {multi['latency']:.1f}s  (coordination overhead -- a real tradeoff)")

Single-agent peak context:   80,059 tokens
Multi-agent PARENT context:  1,056 tokens
(sub-agent work stayed isolated, never in parent: 81,184 tokens total)
Parent window is 99% smaller than the single agent's.

Single latency:  38.1s
Multi latency:   69.7s  (coordination overhead -- a real tradeoff)


## What this experiment proved

1. The single agent's context held ALL exploration for all three areas at once.
2. The multi-agent PARENT held only three short summaries -- its context stayed
   small because each sub-agent's raw work never reached it (Section 8.3).
3. This is compression by ARCHITECTURE: a sub-agent is, to the parent, just a
   tool call that returns a summary.
4. Honest tradeoff: multi-agent is often SLOWER (coordination overhead). Use it
   when isolation/quality matters more than latency.

## Next experiment
Experiment 6 (Tool Explosion): too many tools and raw tool outputs flood a
single agent's context -- and how to tame it.